# Intelli-Speech — EXP2_STAGE_A (measurement validity)

This notebook runs Stage A of Experiment 2 (plan v0.2.1). It checks, on **typical Telugu speech only**, whether the label-free articulation and pause measures are valid and robust to the phone channel.

**What this notebook does not do:**
- Compare atypical with typical speakers (that is Stage B, not implemented).
- Diagnose any speaker or condition.
- Apply any synthetic articulation transform.

**Runtime:** GPU (T4) recommended for cell 5 (alignment); everything else runs on CPU.

**Outputs:** `Intelli-Speech-data/analysis_v2/exp2_stage_a/`. Speakers appear only as T1… or S1… in printed output. **Do not commit this notebook with outputs.**

In [ ]:
# CELL 1 | LOGIN + SYNC + DEPENDENCIES
from huggingface_hub import notebook_login, whoami
try:
    whoami()
except Exception:
    notebook_login()
import os, subprocess, sys, json, shutil
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "https://github.com/niharika9nx/Intelli-Speech.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
assert Path("scripts/exp2_stage_a.py").is_file(), "Pull the latest code first."
!pip install -q datasets soundfile praat-parselmouth "librosa==0.10.2.post1" "numpy<2.5"
!pip install -q pyworld || echo "pyworld install failed -> detectability will be reported as incomplete"
DATA_ROOT = "/content/drive/MyDrive/Intelli-Speech-data"
HF_CACHE = f"{DATA_ROOT}/hf_cache"
OUT = Path(DATA_ROOT) / "analysis_v2" / "exp2_stage_a"
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
enc = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
print("device:", DEVICE, "| libopus:", "libopus" in enc, "| libopencore_amrnb:", "libopencore_amrnb" in enc)
try:
    import torchaudio; print("torchaudio", torchaudio.__version__, "| MMS_FA available:", hasattr(torchaudio.pipelines, "MMS_FA"))
except Exception as e:
    print("torchaudio unavailable:", e)
RUN = f'python scripts/exp2_stage_a.py --data-root "{DATA_ROOT}" --hf-cache "{HF_CACHE}" --device {DEVICE}'

In [ ]:
# CELL 2 | SMOKE TEST (5 clips per phase, separate folder; safe to re-run)
SMOKE = Path(DATA_ROOT) / "analysis_v2" / "exp2_stage_a_smoke"
!{RUN} --output-dir "{SMOKE}" --phase measure --limit 5 --overwrite
!{RUN} --output-dir "{SMOKE}" --phase synthetic --overwrite
!{RUN} --output-dir "{SMOKE}" --phase channel --limit 5 --overwrite
!{RUN} --output-dir "{SMOKE}" --phase align --limit 5 --overwrite

In [ ]:
# CELL 3 | MEASURE typical clips (controls_v2 + reference sample; ~10-20 min CPU)
!{RUN} --phase measure

In [ ]:
# CELL 4 | SYNTHETIC RECOVERY + CHANNEL ROBUSTNESS (~5-15 min CPU)
!{RUN} --phase synthetic
!{RUN} --phase channel

In [ ]:
# CELL 5 | MMS FORCED ALIGNMENT of verbatim typical transcripts (GPU recommended; downloads ~1.2 GB model once)
!{RUN} --phase align

In [ ]:
# CELL 6 | TRANSFORM DETECTABILITY (needs pyworld; reported as incomplete if missing) (~10-20 min CPU)
!{RUN} --phase detectability

In [ ]:
# CELL 7 | EXPORT THE 30-UTTERANCE MANUAL ALIGNMENT CHECK (never overwritten)
!{RUN} --phase manual-sheet
print("Manual-check folder:", OUT / "manual_check")

### Manual alignment check (done by a native Telugu speaker, outside Colab)

1. **Download.** From Drive, download `analysis_v2/exp2_stage_a/manual_check/`.
2. **Open each utterance in Praat.** For each `utt_id`, open its `.wav` and `.TextGrid` together: select both, then **View & Edit**.
   - The **words** tier shows the transcript words.
   - The **vowels** tier shows the automatic vowel boundaries and their category (a, aa, i, ii, …).
3. **Fill `tokens_sheet.csv`, one row per listed vowel token.**
   - `human_start_s`, `human_end_s`:
     - type `=` if the automatic boundary is correct;
     - otherwise type the corrected time in seconds;
     - leave both empty only if you did not check that row.
   - `vowel_present`: `y` if that stretch really contains the vowel, else `n`.
   - `category_correct`: `y` if the vowel is the labelled category, including short vs long, else `n`.
4. **Fill `utterances_sheet.csv`.** `gross_misalignment`: `y` if any word is misplaced by more than its own length, else `n`.
5. **Upload.** Put the two filled sheets back in the same Drive folder, keeping the same file names.
6. **Run cell 8.**

**Acceptance is pre-registered and will not be changed:**
- median absolute boundary error ≤ 30 ms;
- at least 80 % of boundaries within 50 ms;
- utterance failure rate ≤ 20 %;
- category agreement ≥ 85 %.

In [ ]:
# CELL 8 | MANUAL CHECK SUMMARY (run after uploading the filled sheets)
!{RUN} --phase manual-summary --overwrite   # derived output; re-run after uploading the filled sheets

In [ ]:
# CELL 9 | LEXICAL COMPARABILITY (text only) + ATYPICAL FEASIBILITY (sealed: ICCs and counts only)
!{RUN} --phase lexical
!{RUN} --phase atypical-feasibility

In [ ]:
# CELL 10 | STAGE A REPORT
!{RUN} --phase report --overwrite
import pandas as pd
rep = json.loads((OUT / "stage_a_report.json").read_text())
rows = []
for m, e in rep["measures"].items():
    rows.append({"measure": m, "overall": e["overall"], **{t: v.get("status") for t, v in e["tests"].items()}})
print(pd.DataFrame(rows).fillna("—").to_string(index=False))
print(); print("manual alignment:", (rep.get("manual_alignment") or {}).get("acceptance"))
print("lexical flag:", (rep.get("lexical_flag") or {}).get("flag"))

Save the notebook and tell Claude. Do not commit it with outputs.